# Week 1 Day 2 assignment - a free model showdown

## Which free model decides UnderCover's claims best, and is an Agent worth it?

This is my take on the Day 2 assignment from [week1day2free.ipynb](../../labs/week1day2free.ipynb). That lab sends claims to `openrouter/free`, which picks a different free model each time. The assignment asks: use a *specific* free model instead, and find which one gets the most claims right with the agent.

I went a bit wider and put two free providers side by side:

- **OpenRouter free**: `nvidia/nemotron-3-super-120b-a12b:free`
- **Ollama cloud free**: seven models, from a 20B up to a 550B

Every model decides the same claims twice, once as a single LLM call and once as a LangChain agent with tools, and the handlers' decisions are the ground truth. So we get accuracy and speed for each model *and* for each technique.

## 4 parts:

PART 1: The data

PART 2: One call per claim

PART 3: The same job, with an Agent

PART 4: The scoreboard

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This notebook needs <b>two</b> free accounts, and no credit on either.<br/><br/>
<b>1. OpenRouter.</b> The same <code>OPENROUTER_API_KEY</code> in the <code>.env</code> file at the root of the <code>undercover</code> repo, from day 1. An account with no credit allows 50 requests a day, and this notebook spends about 25 of them. Run it twice in a day and the second run will hit the cap: the scored runs record that as a miss and carry on, but the two single-claim demo cells will raise.<br/><br/>
<b>2. Ollama cloud.</b> Install <a href="https://ollama.com/download">Ollama</a>, run <code>ollama signin</code>, then pull the seven models with the cell below. Cloud models run on Ollama's hardware, so nothing is downloaded and a laptop can run a 550B model; the free tier is rate limited by the hour and by the day.<br/><br/>
With <code>TINY = True</code>, the default here, this notebook makes about 140 Ollama requests. If you hit a limit, drop models from <code>OLLAMA_MODELS</code> or come back in an hour. Every call is wrapped in a <code>try</code>, so one failure scores as a miss instead of ending the run.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Pull the seven Ollama cloud models. Each one is just a manifest, so this is quick and uses no disk.
# Run ollama signin first if you haven't.

!ollama pull beam:cloud
!ollama pull gemma4:31b-cloud
!ollama pull gpt-oss:120b-cloud
!ollama pull gpt-oss:20b-cloud
!ollama pull nemotron-3-nano:30b-cloud
!ollama pull nemotron-3-super:cloud
!ollama pull nemotron-3-ultra:cloud

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/resources.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Where this came from</h2>
            <span style="color:#118AB2;">This is a community contribution built on Ed Donner's Week 1 Day 2 lab. The data, the policy wording and the handlers' decisions all come from the <code>data</code> folder in the repo, so the scores here can be compared with the lab's.<br/><br/>
The <a href="../../guides/01_intro.ipynb">guides</a> folder has more on the command line, Git, Python and debugging.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI

## Connect to the two free providers

Both speak the Chat Completions API, so one `OpenAI` client each is enough and the rest of the notebook doesn't care which provider a model is on. `client_for` picks by the model id: anything ending in `:free` is OpenRouter, everything else is Ollama.

Ollama needs no key, but the client insists on one, so it gets the string `"ollama"`.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

OPENROUTER_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"
OLLAMA_MODELS = [
    "beam:cloud",
    "gemma4:31b-cloud",
    "gpt-oss:120b-cloud",
    "gpt-oss:20b-cloud",
    "nemotron-3-nano:30b-cloud",
    "nemotron-3-super:cloud",
    "nemotron-3-ultra:cloud",
]
MODELS = [OPENROUTER_MODEL] + OLLAMA_MODELS

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=300, max_retries=3)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=600, max_retries=2)


def client_for(model):
    """OpenRouter for the :free models, Ollama for everything else."""
    return openrouter if model.endswith(":free") else ollama


print("OpenRouter API key found" if openrouter_api_key else "No OPENROUTER_API_KEY in .env")
print(f"{len(MODELS)} models: 1 on OpenRouter, {len(OLLAMA_MODELS)} on Ollama")

### Roll call

Before spending any of the daily budget on claims, check that all eight models answer. If one fails here it will fail later, so drop it from the list rather than let it score zero.

In [ ]:
def say_hello(model):
    """One tiny call, to prove the model is reachable."""
    start = time.time()
    response = client_for(model).chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "In under 10 words: what is travel insurance for?"}],
    )
    return response.choices[0].message.content.strip(), round(time.time() - start, 1)


for model in MODELS:
    try:
        reply, seconds = say_hello(model)
        print(f"{model:35s} {seconds:5.1f}s  {reply[:60]}")
    except Exception as error:
        print(f"{model:35s} FAILED  {type(error).__name__}: {error}"[:110])

## Part 1: The data

Same seed data as the lab, read from the repo's `data` folder two levels up. `get_sample_data` returns the claims and the handlers' decisions on them, keyed by claim id.

`TINY = True` keeps 5 of the 25 claims. With eight models and two techniques that is already 80 decisions, so leave it on unless you have budget to burn.

In [ ]:
DATA = Path("../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id."""
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")
print(f"{len(MODELS)} models x {len(claims)} claims x 2 techniques = {len(MODELS) * len(claims) * 2} decisions")

Here is what the models are up against. `difficulty` is the handler's own label: `routine` and `hard` can both be decided from the policy wording, while `manual` needs the claims manual, which nobody gets to see in this notebook. So every model is expected to miss the manual ones.

In [ ]:
data = pd.DataFrame([
    {
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": labels[claim["claim_id"]]["difficulty"],
        "decision": labels[claim["claim_id"]]["decision"],
        "payout": labels[claim["claim_id"]]["payout"],
    }
    for claim in claims
])

data

## Part 2: One call per claim

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Why a tool call and not Structured Outputs</h2>
            <span style="color:#118AB2;">The lab uses Structured Outputs: a Pydantic class goes in as <code>response_format</code> and a parsed object comes back. <b>Ollama's cloud models quietly ignore it.</b> Pass a schema as <code>response_format</code> or as Ollama's own <code>format</code> and you still get prose back, so <code>parse()</code> raises a validation error on the first claim.<br/><br/>
They all support <b>tool calling</b>, though, and so does the OpenRouter model. So instead of asking for JSON, we give every model one tool, <code>submit_decision</code>, and set <code>tool_choice="required"</code> so it has to call it. The arguments come back as JSON against our schema, which is what Structured Outputs was for.<br/><br/>
It also makes Part 2 and Part 3 a fair fight: both are now the model filling in the same tool, once in a single call and once inside an agent loop.
            </span>
        </td>
    </tr>
</table>

The system prompt is the lab's, with one line added about the tool, and the whole policy wording. The user prompt has the customer's policy record and the claim, both as JSON.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Call submit_decision exactly once with your decision. Do not reply in prose.

{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

The tool schema is the lab's `ClaimDecision` class written out by hand. `rationale` comes first so the model reasons before it commits to a decision.

`tidy` is here because the schema's `enum` is a suggestion, not a rule, on these models. One returned `"Approved"` and another invented `"partial"`. Lowercasing and matching on the first six letters catches the first; the second is simply a wrong answer and is left alone to score as a miss.

In [ ]:
SUBMIT_DECISION = [{
    "type": "function",
    "function": {
        "name": "submit_decision",
        "description": "Submit your decision on the claim.",
        "parameters": {
            "type": "object",
            "properties": {
                "rationale": {"type": "string", "description": "2-3 sentences to explain the decision along with the relevant section of the policy"},
                "decision": {"type": "string", "enum": ["approve", "decline", "refer"]},
                "payout": {"type": "number", "description": "The amount to pay in US dollars, or 0 for a decline or a referral"},
            },
            "required": ["rationale", "decision", "payout"],
        },
    },
}]


def tidy(decision):
    """Models don't always honour the enum: "Approved" becomes "approve", "partial" stays wrong."""
    word = str(decision).strip().lower()
    for choice in ("approve", "decline", "refer"):
        if word.startswith(choice[:6]):
            return choice
    return word

In [ ]:
def decide(claim, model):
    """Ask one model to decide one claim, in a single call."""
    start = time.time()
    response = client_for(model).chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": make_user_prompt(claim)},
        ],
        tools=SUBMIT_DECISION,
        tool_choice="required",
    )
    calls = response.choices[0].message.tool_calls
    answer = json.loads(calls[0].function.arguments) if calls else {}
    return {
        "model": model,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": tidy(answer.get("decision", "nothing submitted")),
        "payout": float(answer.get("payout") or 0),
        "rationale": answer.get("rationale", ""),
        "seconds": round(time.time() - start, 1),
    }

One claim first, to see the shape of an answer.

In [ ]:
# decide(claims[0], OPENROUTER_MODEL)

### Scoring

A decision is right when it matches the handler's and the payout is within a dollar. `run` walks the claims for one model and catches anything that goes wrong, so a rate limit or a timeout scores as a miss and the next model still gets its turn.

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def failed(claim, model, technique, error):
    """A call that didn't come back, scored as a miss."""
    return {
        "model": model, "technique": technique, "claim_id": claim["claim_id"],
        "decision": "error", "payout": 0.0, "rationale": f"{type(error).__name__}: {error}"[:300], "seconds": 0.0,
    }


def run(model, decider=decide, technique="LLM"):
    """Run one model over the claims and compare against the handlers' decisions."""
    results = []
    for claim in claims:
        label = labels[claim["claim_id"]]
        try:
            answer = decider(claim, model)
        except Exception as error:
            answer = failed(claim, model, technique, error)
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model:35s} {technique:5s} {right} of {len(results)} right in {seconds:5.0f} seconds")
    return results

Now all eight models, one call per claim. This takes a few minutes, mostly waiting on the big models.

In [ ]:
llm_results = []
for model in MODELS:
    llm_results += run(model)

In [ ]:
llm = pd.DataFrame(llm_results)
llm.pivot_table(index="model", values=["right", "seconds"], aggfunc="sum").sort_values("right", ascending=False)